#### Transform Drivers Data (Bronze to Silver)

**Steps:**
1. Read the raw data from the Bronze table
2. Drop the `url` column
3. Rename `driverId` to `driver_id` and `dateOfBirth` to `date_of_birth`
4. Concatenate `name.givenName` and `name.familyName` into a new `driver_name` column in title case
5. Drop the nested `name` column
6. Remove duplicates based on `driver_id`
7. Apply title case to `nationality`
8. Save to Silver table

#### Loading Configuration and Setting Variables

In [0]:
%run ../00-common/01.environment-config 

In [0]:
bronze_table = f'{catalog_name}.{bronze_schema}.drivers'
silver_table = f'{catalog_name}.{silver_schema}.drivers'

#### Read Bronze Table
Read the raw drivers data from `formula1.bronze.drivers` into a DataFrame.

In [0]:
drivers_df = spark.read.table(bronze_table)


##### Keep only the column that is required for Analysis 
- Drop (url)

#### Drop Columns
Remove the `url` column - not needed for analysis.

In [0]:
drivers_drop_df = drivers_df.drop('url')

#### Rename Columns
Rename `driverId` to `driver_id` and `dateOfBirth` to `date_of_birth` for consistent snake_case naming.

In [0]:
from pyspark.sql import functions as F
drivers_renamed_df = drivers_drop_df.withColumnRenamed('driverId', 'driver_id').withColumnRenamed('dateOfBirth', 'date_of_birth')                    

In [0]:
display(drivers_renamed_df)

#### Concatenate Names
Combine `name.givenName` and `name.familyName` into a single `driver_name` column using `concat_ws()`, apply `initcap()` for title case, then drop the original nested `name` column.

In [0]:
from pyspark.sql.functions import *
drivers_concatenate_df =(
    drivers_renamed_df
    .withColumn('driver_name',
              initcap(concat_ws(' ', col('name.givenName'), F.col('name.FamilyName'))))
    .drop('name')
)
display(drivers_concatenate_df)

#### Remove Duplicates
Drop duplicate rows based on `driver_id` column.

In [0]:
drivers_distinct_df = drivers_concatenate_df.dropDuplicates(['driver_id'])


#### Title Case
Convert `nationality` to title case using `initcap()` (e.g., "british" becomes "British").

In [0]:
drivers_final_df = (drivers_distinct_df.withColumn('nationality', initcap(col('nationality'))))
display(drivers_final_df)

#### Write to Silver Table
Save the cleaned DataFrame to `formula1.silver.drivers` in Delta format with overwrite mode.

In [0]:
(
    drivers_final_df.write
    .mode('overwrite')
    .format('delta')
    .saveAsTable(silver_table)
)

In [0]:
spark.read.table(silver_table).display() 